# Install Required Libraries

--- 

In [1]:
!pip install boto3 awswrangler pyathena seaborn

## Connect to S3

In [2]:
import boto3
import pandas as pd
import awswrangler as wr

bucket = "prognostica-cancer-project"
raw_path = f"s3://{bucket}/raw/"
processed_path = f"s3://{bucket}/processed/"

print(raw_path)

s3://prognostica-cancer-project/raw/


## Load the Datasets from S3

In [6]:
bucket = "prognostica-cancer-project"
prefix = "raw/"

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    print(obj["Key"])

raw/
raw/TEQ_2020.csv
raw/TEQ_2021.csv
raw/TEQ_2022.csv
raw/TEQ_2023.csv
raw/TEQ_2024.csv
raw/United States and Puerto Rico Cancer Statistics, 1999-2022 Incidence.csv
raw/lcs_synthetic_20000.csv


## Lung Cancer Dataset

In [7]:
lung_df = wr.s3.read_csv(f"{raw_path}lcs_synthetic_20000.csv")

lung_df.head()

,GENDER,AGE,SMOKING,YELLOW_FINGERS,ANXIETY,PEER_PRESSURE,CHRONIC DISEASE,FATIGUE,ALLERGY,WHEEZING,ALCOHOL CONSUMING,COUGHING,SHORTNESS OF BREATH,SWALLOWING DIFFICULTY,CHEST PAIN,LUNG_CANCER
0,M,69,2,1,1,2,1,2,1,1,2,2,2,1,1,YES
1,M,71,2,2,1,1,2,1,2,2,1,1,2,2,1,YES
2,M,61,2,1,1,2,2,1,2,2,1,1,2,2,2,NO
3,M,55,2,2,1,2,1,1,1,2,2,1,2,2,2,YES
4,F,56,2,1,1,1,1,2,2,2,2,1,2,2,2,YES


In [8]:
lung_df.shape
lung_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   GENDER                 20000 non-null  object
 1   AGE                    20000 non-null  int64 
 2   SMOKING                20000 non-null  int64 
 3   YELLOW_FINGERS         20000 non-null  int64 
 4   ANXIETY                20000 non-null  int64 
 5   PEER_PRESSURE          20000 non-null  int64 
 6   CHRONIC DISEASE        20000 non-null  int64 
 7   FATIGUE                20000 non-null  int64 
 8   ALLERGY                20000 non-null  int64 
 9   WHEEZING               20000 non-null  int64 
 10  ALCOHOL CONSUMING      20000 non-null  int64 
 11  COUGHING               20000 non-null  int64 
 12  SHORTNESS OF BREATH    20000 non-null  int64 
 13  SWALLOWING DIFFICULTY  20000 non-null  int64 
 14  CHEST PAIN             20000 non-null  int64 
 15  LUNG_CANCER        

## Cancer Incidence Dataset

In [10]:
incidence_df = wr.s3.read_csv(f"{raw_path}US_cancer_incidence_1999_2022.csv")

incidence_df.head()

,Notes,Leading Cancer Sites,Leading Cancer Sites Code,Count
0,NaN,Brain and Other Nervous System,31010-31040,523196.0
1,NaN,Breast,26000,5558064.0
2,NaN,Cervix Uteri,27010,310804.0
3,NaN,Colon and Rectum,21041-21052,3521960.0
4,NaN,Corpus Uteri,27020,1121662.0


In [11]:
incidence_df.shape
incidence_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Notes                      38 non-null     object 
 1   Leading Cancer Sites       22 non-null     object 
 2   Leading Cancer Sites Code  22 non-null     object 
 3   Count                      22 non-null     float64
dtypes: float64(1), object(3)
memory usage: 2.0+ KB


## Load the TEQ Environmental Files

In [14]:
teq_2020 = wr.s3.read_csv(f"{raw_path}TEQ_2020.csv")
teq_2021 = wr.s3.read_csv(f"{raw_path}TEQ_2021.csv")
teq_2022 = wr.s3.read_csv(f"{raw_path}TEQ_2022.csv")
teq_2023 = wr.s3.read_csv(f"{raw_path}TEQ_2023.csv")
teq_2024 = wr.s3.read_csv(f"{raw_path}TEQ_2024.csv")

print(teq_2020.shape)
print(teq_2021.shape)
print(teq_2022.shape)
print(teq_2023.shape)
print(teq_2024.shape)

(818, 90)
(812, 90)
(813, 90)
(770, 90)
(721, 90)


## Combine the Environmental Datasets

In [15]:
teq_combined = pd.concat([
    teq_2020,
    teq_2021,
    teq_2022,
    teq_2023,
    teq_2024
], ignore_index=True)

teq_combined.shape

(3934, 90)

In [16]:
teq_combined.head()

,Year,TRI Facility ID,Facility Name,Street Address,City,County,ST,ZIP,Latitude,Longitude,...,8.1d Off-site Other Releases,8.2 Energy Recovery On-site,8.3 Energy Recovery Off-site,8.4 Recycling On-Site,8.5 Recycling Off-Site,8.6 Treatment On-site,8.7 Treatement Off-site,8.8 One-time Release,20251105,KLJ
0,2020,00785SPRTRKM142,AES PUERTO RICO LP,KM 142 RD #3 BO. JOBOS,GUAYAMA,GUAYAMA MUNICIPIO,PR,784,17.943889,-66.150917,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0,NaN,NaN
1,2020,00804VRGNSKRUMB,VIRGIN ISLANDS WATER & POWER AUTHORITY,KRUM BAY FACILITY,SAINT THOMAS,ST. THOMAS ISLAND,VI,804,18.330234,-64.960147,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0,NaN,NaN
2,2020,00821VRGNSESTAT,VIRGIN ISLANDS WATER & POWER AUTHORITY,ESTATE RICHMOND,CHRISTIANSTED,ST. CROIX ISLAND,VI,821,17.750141,-64.714793,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0,NaN,NaN
3,2020,00936SNJNCKM267,ARGOS PUERTO RICO CORP.,KM 26.7 STATE HWY #2,DORADO,DORADO MUNICIPIO,PR,646,18.394400,-66.297600,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0,NaN,NaN
4,2020,04239NTRNTRILEY,PIXELLE SPECIALTY SOLUTIONS,300 RILEY RD - ANDROSCOGGIN MILL,JAY,FRANKLIN,ME,4239,44.507100,-70.238810,...,0.0,0.0,0.0,0.0,0.0,0.18291,0.0,0,NaN,NaN


## Save Combined Dataset

In [17]:
teq_combined.to_csv("teq_combined_2020_2024.csv", index=False)

## Upload Combined Dataset to S3 (processed folder)

In [18]:
wr.s3.to_csv(
    teq_combined,
    f"{processed_path}teq_combined_2020_2024.csv",
    index=False
)

{'paths': ['s3://prognostica-cancer-project/processed/teq_combined_2020_2024.csv'],
 'partitions_values': {}}

# Data Exploration

## Dataset Overview

In [21]:
lung_df.describe()
incidence_df.describe()
teq_combined.describe()

,Year,ZIP,Latitude,Longitude,Primary NAICS,NAICS 2,NAICS 3,NAICS 4,NAICS 5,NAICS 6,...,8.1d Off-site Other Releases,8.2 Energy Recovery On-site,8.3 Energy Recovery Off-site,8.4 Recycling On-Site,8.5 Recycling Off-Site,8.6 Treatment On-site,8.7 Treatement Off-site,8.8 One-time Release,20251105,KLJ
count,3934.000000,3.934000e+03,3934.000000,3934.000000,3934.000000,566.000000,131.000000,46.000000,5.000000,0.0,...,3934.000000,3934.000000,3934.000000,3934.000000,3934.000000,3934.000000,3934.000000,3934.0,0.0,0.0
mean,2021.940010,7.435481e+06,37.399353,-92.321871,304514.577021,318243.948763,336896.580153,312629.934783,326900.800000,NaN,...,0.036782,0.000289,0.000857,0.046545,0.004240,1.329399,0.141179,0.0,NaN,NaN
std,1.401376,7.385440e+07,5.874494,16.097219,74209.371246,65311.174531,72963.978290,70297.115462,961.956444,NaN,...,1.135886,0.015842,0.013307,1.022996,0.098593,19.606449,3.823397,0.0,NaN,NaN
min,2020.000000,6.460000e+02,13.464516,-162.855492,211130.000000,212111.000000,113310.000000,212230.000000,325180.000000,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN
25%,2021.000000,3.590300e+04,33.227060,-96.348230,221112.000000,322110.000000,322251.000000,321212.000000,327331.000000,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN
50%,2022.000000,5.445700e+04,37.539200,-89.644665,322130.000000,325110.000000,325199.000000,325125.000000,327331.000000,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN
75%,2023.000000,7.557200e+04,41.269767,-83.523750,327310.000000,331314.000000,329317.000000,325998.000000,327331.000000,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,NaN,NaN
max,2024.000000,9.970223e+08,68.061507,144.657067,928110.000000,562213.000000,562910.000000,562998.000000,327331.000000,NaN,...,54.943675,0.992251,0.466977,32.436994,3.006624,775.658295,227.182115,0.0,NaN,NaN


## Missing Values

In [22]:
lung_df.isnull().sum()
incidence_df.isnull().sum()
teq_combined.isnull().sum()

Year                          0
TRI Facility ID               0
Facility Name                 0
Street Address                0
City                          0
                           ... 
8.6 Treatment On-site         0
8.7 Treatement Off-site       0
8.8 One-time Release          0
20251105                   3934
KLJ                        3934
Length: 90, dtype: int64